# Complete-case (missingness) sensitivity

Same incident BASE / CLIN / SOC (+NMR) ridge Cox models as `primary_incident_ckm.ipynb`, with one change to the SOC marker block:

1. Restrict the cohort with the missingness indicator (`M_* == 1`), so only people with an available measurement remain.
2. Drop that missingness indicator from the model. Keep the observed value and the time-delay (staleness) term.

No PRS specs. Incident follow-up is `time_col > 0`. `fit_full` stays `FALSE`. Fold models and absolute-risk objects are not saved.

Outputs stay out of `outputs/main`:

```
outputs/supp/complete_case/<tag>_<spec>_oof.rds
outputs/tables/CINDEX_STATS_COMPLETE_CASE.csv
outputs/figures/COMPLETE_CASE.png
outputs/figures/COMPLETE_CASE_delta.png
```

Existing `.rds` files are skipped (`skip_existing = TRUE`).


In [ ]:
suppressPackageStartupMessages({
  library(ggpubr)
  library(patchwork)
  library(tidyverse)
  library(survival)
  library(glmnet)
  library(here)
})


# Load data


In [ ]:
source(here("helpers", "prep_incident_time_to_event_df.r"))
source(here("helpers", "summarize_time_to_event.r"))
source(here("helpers", "cv_glmnet_oof.r"))
source(here("helpers", "fit_full_cox.r"))

meta <- read.csv(here("data", "meta_clin.csv"), row.names = 1) %>%
  dplyr::mutate(StudyID = as.character(StudyID))
assay <- read.csv(here("data", "assay.csv"), row.names = 1) %>%
  dplyr::mutate(StudyID = as.character(StudyID))
nmr_names <- assay %>% select(-StudyID) %>% names()

cat("meta dimensions:", dim(meta), "\n")
cat("assay dimensions:", dim(assay), "\n")


# Configuration


In [ ]:
cv_defaults <- list(
  outer_nfolds   = 5,
  inner_nfolds   = 10,
  nlambda        = 100,
  seed           = 1,
  standardize    = FALSE,
  parallel_outer = TRUE,
  n_cores        = 5L,
  alpha          = 0
)

common_blocks <- list(
  base = c("age_at_sample", "sex"),
  clin = c("BMI_interp", "SBP_interp"),
  nmr  = nmr_names
)

common_scale_blocks <- list(
  base = c("age_at_sample"),
  clin = c("BMI_interp", "SBP_interp"),
  nmr  = nmr_names
)

model_specs <- list(
  base       = c("base"),
  base_nmr   = c("base", "nmr"),
  clin       = c("base", "clin"),
  clin_nmr   = c("base", "clin", "nmr"),
  marker     = c("base", "clin", "marker"),
  marker_nmr = c("base", "clin", "marker", "nmr")
)

COMPLETE_CASE_ROOT <- here("outputs/supp/complete_case")
FIG_ROOT <- here("outputs/figures")
dir.create(file.path(COMPLETE_CASE_ROOT, "c_index_summaries"), recursive = TRUE, showWarnings = FALSE)
dir.create(FIG_ROOT, recursive = TRUE, showWarnings = FALSE)


In [ ]:
resolve_blocks <- function(block_names, block_map) {
  missing_blocks <- setdiff(block_names, names(block_map))
  if (length(missing_blocks) > 0) {
    stop("Missing block definitions: ", paste(missing_blocks, collapse = ", "))
  }
  unique(unlist(block_map[block_names], use.names = FALSE))
}

make_penalty_factor <- function(predictors, unpenalized = character()) {
  pf <- rep(1, length(predictors))
  names(pf) <- predictors
  pf[intersect(unpenalized, predictors)] <- 0.1
  pf
}

make_model_maps <- function(marker_predictors, marker_scale = character()) {
  list(
    blocks = c(common_blocks, list(marker = marker_predictors)),
    scale_blocks = c(common_scale_blocks, list(marker = marker_scale))
  )
}

get_required_vars <- function(maps,
                              specs = c("base", "clin", "marker"),
                              extra_vars = c("time", "status"),
                              exclude_blocks = c("nmr")) {
  used_blocks <- unique(unlist(model_specs[specs], use.names = FALSE))
  used_blocks <- setdiff(used_blocks, exclude_blocks)
  unique(c(extra_vars, resolve_blocks(used_blocks, maps$blocks)))
}

filter_complete_cases_for_specs <- function(df, maps,
                                            specs = c("base", "clin", "marker"),
                                            extra_vars = c("time", "status"),
                                            exclude_blocks = c("nmr")) {
  vars_needed <- get_required_vars(
    maps = maps,
    specs = specs,
    extra_vars = extra_vars,
    exclude_blocks = exclude_blocks
  )
  df %>% filter(if_all(all_of(vars_needed), ~ !is.na(.)))
}

apply_case_filter <- function(df, expr_txt) {
  ex <- rlang::parse_expr(expr_txt)
  dplyr::filter(df, !!ex)
}

print_cv_summary <- function(results) {
  cat(sprintf("\nOOF C-index: %.4f (SE: %.4f)\n", results$c_index, results$se))
  cat(sprintf("95%% CI: [%.4f, %.4f]\n", results$ci_lower, results$ci_upper))
}

run_cv_spec <- function(df, spec_name, maps,
                        unpenalized = character(),
                        output_path = NULL,
                        model_engine = "glmnet",
                        fit_full = FALSE,
                        keep_fold_models = FALSE) {
  if (!spec_name %in% names(model_specs)) {
    stop("Unknown spec_name: ", spec_name)
  }

  block_names   <- model_specs[[spec_name]]
  predictors    <- resolve_blocks(block_names, maps$blocks)
  vars_to_scale <- resolve_blocks(block_names, maps$scale_blocks)

  missing_predictors <- setdiff(c("time", "status", predictors), names(df))
  if (length(missing_predictors) > 0) {
    stop(sprintf("Spec '%s' requested variables not found in df: %s",
                 spec_name, paste(missing_predictors, collapse = ", ")))
  }
  missing_scale <- setdiff(vars_to_scale, names(df))
  if (length(missing_scale) > 0) {
    stop(sprintf("Spec '%s' vars_to_scale not found in df: %s",
                 spec_name, paste(missing_scale, collapse = ", ")))
  }

  df_model <- df %>%
    dplyr::select(dplyr::all_of(c("StudyID", "time", "status", predictors))) %>%
    dplyr::filter(dplyr::if_all(dplyr::all_of(c("time", "status", predictors)), ~ !is.na(.)))

  unpenalized <- intersect(unpenalized, predictors)
  penalty.factor <- make_penalty_factor(predictors, unpenalized)

  args <- c(
    list(
      df = df_model,
      predictors = predictors,
      vars_to_scale = vars_to_scale,
      penalty.factor = penalty.factor,
      model_engine = model_engine
    ),
    cv_defaults
  )
  if ("nmr" %in% block_names) {
    args$parallel_outer <- FALSE
  }
  results <- do.call(cv_glmnet_cox_oof, args)
  print_cv_summary(results)

  if (!isTRUE(keep_fold_models)) {
    results$fold_models <- NULL
  }
  invisible(gc(verbose = FALSE))

  if (isTRUE(fit_full)) {
    full_args <- c(
      list(
        df             = df_model,
        predictors     = predictors,
        vars_to_scale  = vars_to_scale,
        penalty.factor = penalty.factor,
        model_engine   = model_engine
      ),
      cv_defaults[setdiff(names(cv_defaults), c("outer_nfolds", "parallel_outer", "n_cores"))]
    )
    results$full_model <- do.call(fit_full_cox, full_args)
  }

  if (!is.null(output_path)) {
    saveRDS(results, output_path)
  }
  invisible(results)
}


## Complete-case filters and marker maps

Each block is the instruction for that endpoint. The filter keeps measured labs. The marker map is what the SOC model then sees: observed value and time delay, without `M_*`.

PREVENT endpoints (CHD, heart failure, acute MI, TIA/stroke, total CVD) share one filter and one map.


In [ ]:
# T2D
# filter: M_a1c == 1L
# dropped from the model: M_a1c
t2d_maps <- make_model_maps(
  marker_predictors = c("A1C_obs", "T_a1c"),
  marker_scale      = c("A1C_obs", "T_a1c")
)

# CKD
# filter: M_egfr == 1L
# dropped from the model: M_egfr
ckd_maps <- make_model_maps(
  marker_predictors = c("EGFR_obs", "T_egfr"),
  marker_scale      = c("EGFR_obs", "T_egfr")
)

# MASLD
# filter: M_fib4 == 1L
# dropped from the model: M_fib4
nafld_maps <- make_model_maps(
  marker_predictors = c("FIB4_obs", "T_fib4"),
  marker_scale      = c("FIB4_obs", "T_fib4")
)

# COPD
# filter: M_smoke == 1L & M_ldl == 1L
# dropped from the model: M_smoke, M_ldl
copd_maps <- make_model_maps(
  marker_predictors = c("smk_current", "smk_former", "LDL_obs", "T_ldl"),
  marker_scale      = c("LDL_obs", "T_ldl")
)

# CHD, heart failure, acute MI, TIA/stroke, total CVD
# filter: M_tot_choles == 1L & M_hdl_choles == 1L & M_egfr == 1L & M_smoke == 1L
# dropped from the model: M_tot_choles, M_hdl_choles, M_egfr, M_smoke
chd_complete_maps <- make_model_maps(
  marker_predictors = c(
    "TOT_CHOLES_obs", "T_tot_choles",
    "HDL_CHOLES_obs", "T_hdl_choles",
    "EGFR_obs", "T_egfr",
    "smk_current", "smk_former",
    "bp_treatment", "statin_treatment", "diabetes_at_baseline"
  ),
  marker_scale = c(
    "TOT_CHOLES_obs", "T_tot_choles",
    "HDL_CHOLES_obs", "T_hdl_choles",
    "EGFR_obs", "T_egfr"
  )
)

# Atrial fibrillation
# filter: M_smoke_af == 1L
# dropped from the model: M_smoke_af
# New meta_clin names: smoking_current_af, height_interp, weight_interp, DBP_interp,
# hf_at_baseline, mi_at_baseline (not the older CHARGE-AF column names).
afib_complete_maps <- make_model_maps(
  marker_predictors = c(
    "smoking_current_af",
    "height_interp", "weight_interp", "DBP_interp",
    "bp_treatment", "hf_at_baseline", "mi_at_baseline",
    "diabetes_at_baseline", "race"
  ),
  marker_scale = c("height_interp", "weight_interp", "DBP_interp")
)

endpoint_config <- list(
  t2d = list(
    label = "T2D", plot_label = "Type 2 Diabetes", tag = "t2d_a1c",
    status_col = "t2d_status", time_col = "t2d_time",
    case_filter = "M_a1c == 1L",
    maps = t2d_maps
  ),
  ckd = list(
    label = "CKD", plot_label = "Chronic Kidney Disease", tag = "ckd_egfr",
    status_col = "ckd_status", time_col = "ckd_time",
    case_filter = "M_egfr == 1L",
    maps = ckd_maps
  ),
  masld = list(
    label = "MASLD", plot_label = "MASLD (Liver Disease)", tag = "nafld_fib4",
    status_col = "nafld_status", time_col = "nafld_time",
    case_filter = "M_fib4 == 1L",
    maps = nafld_maps
  ),
  copd = list(
    label = "COPD", plot_label = "Chronic Obstructive Pulmonary Disease", tag = "copd_smoking_ldl",
    status_col = "copd_status", time_col = "copd_time",
    case_filter = "M_smoke == 1L & M_ldl == 1L",
    maps = copd_maps
  ),
  chd_revasc = list(
    label = "CHD", plot_label = "Coronary Heart Disease", tag = "chd_revasc_prevent_ingredients",
    status_col = "chd_minerva_status", time_col = "chd_minerva_time",
    case_filter = "M_tot_choles == 1L & M_hdl_choles == 1L & M_egfr == 1L & M_smoke == 1L",
    maps = chd_complete_maps
  ),
  hf_cm = list(
    label = "Heart Failure", plot_label = "Heart Failure", tag = "hf_cm_prevent_ingredients",
    status_col = "hf_cm_status", time_col = "hf_cm_time",
    case_filter = "M_tot_choles == 1L & M_hdl_choles == 1L & M_egfr == 1L & M_smoke == 1L",
    maps = chd_complete_maps
  ),
  acute_mi = list(
    label = "Acute MI", plot_label = "Acute MI", tag = "acute_mi_prevent_ingredients",
    status_col = "acute_mi_status", time_col = "acute_mi_time",
    case_filter = "M_tot_choles == 1L & M_hdl_choles == 1L & M_egfr == 1L & M_smoke == 1L",
    maps = chd_complete_maps
  ),
  tia_stroke = list(
    label = "TIA/Stroke", plot_label = "TIA/Stroke", tag = "tia_stroke_prevent_ingredients",
    status_col = "tia_stroke_status", time_col = "tia_stroke_time",
    case_filter = "M_tot_choles == 1L & M_hdl_choles == 1L & M_egfr == 1L & M_smoke == 1L",
    maps = chd_complete_maps
  ),
  total_cvd = list(
    label = "Total CVD", plot_label = "Total CVD", tag = "total_cvd_prevent_ingredients",
    status_col = "total_cvd_status", time_col = "total_cvd_time",
    case_filter = "M_tot_choles == 1L & M_hdl_choles == 1L & M_egfr == 1L & M_smoke == 1L",
    maps = chd_complete_maps
  ),
  afib = list(
    label = "Afib", plot_label = "Atrial fibrillation", tag = "afib_chargeaf_ingredients",
    status_col = "afib_status", time_col = "afib_time",
    case_filter = "M_smoke_af == 1L",
    maps = afib_complete_maps
  )
)

# Printed checklist of the instructions above.
case_checklist <- dplyr::bind_rows(lapply(names(endpoint_config), function(nm) {
  cfg <- endpoint_config[[nm]]
  data.frame(
    endpoint = nm,
    case_filter = cfg$case_filter,
    marker_predictors = paste(cfg$maps$blocks$marker, collapse = ", "),
    stringsAsFactors = FALSE
  )
}))
case_checklist


In [ ]:
oof_path <- function(tag, spec) {
  file.path(COMPLETE_CASE_ROOT, paste0(tag, "_", spec, "_oof.rds"))
}

prepare_endpoint_df <- function(nm) {
  cfg <- endpoint_config[[nm]]
  if (is.null(cfg)) stop("Unknown endpoint: ", nm)

  time_filter <- paste(cfg$time_col, "> 0")
  message("Preparing ", nm, " (", cfg$label, ")")
  message("  time_filter: ", time_filter)
  message("  case_filter: ", cfg$case_filter)
  message("  marker predictors: ", paste(cfg$maps$blocks$marker, collapse = ", "))

  meta_use <- prep_incident_time_to_event_df(
    meta,
    status_col  = cfg$status_col,
    time_col    = cfg$time_col,
    time_filter = time_filter
  )
  n_before <- nrow(meta_use)
  meta_use <- apply_case_filter(meta_use, cfg$case_filter)
  message("  n after case filter: ", nrow(meta_use), " (from ", n_before, ")")

  meta_use <- filter_complete_cases_for_specs(
    df = meta_use,
    maps = cfg$maps,
    specs = c("base", "clin", "marker")
  )
  meta_use_nmr <- meta_use %>% inner_join(assay, by = "StudyID")

  cat("analytical n:", nrow(meta_use_nmr),
      " events:", sum(meta_use_nmr$status), "\n")

  list(df = meta_use_nmr, maps = cfg$maps, cfg = cfg)
}

run_endpoint <- function(nm,
                         specs = names(model_specs),
                         skip_existing = TRUE,
                         fit_full = FALSE) {
  cfg <- endpoint_config[[nm]]
  if (is.null(cfg)) stop("Unknown endpoint: ", nm)

  prepared <- prepare_endpoint_df(nm)

  for (sp in specs) {
    path <- oof_path(cfg$tag, sp)
    if (skip_existing && file.exists(path)) {
      message("Skipping existing OOF: ", basename(path))
      next
    }
    cat("\n==============================\n", cfg$tag, "::", sp,
        "\n==============================\n")
    results <- run_cv_spec(
      df           = prepared$df,
      spec_name    = sp,
      maps         = prepared$maps,
      unpenalized  = c("age_at_sample", "sex"),
      model_engine = "glmnet",
      output_path  = path,
      fit_full     = fit_full,
      keep_fold_models = FALSE
    )
    rm(results)
    cleanup_all_clusters()
    gc(verbose = FALSE)
  }

  invisible(list(
    endpoint = nm,
    tag      = cfg$tag,
    n        = nrow(prepared$df),
    n_events = sum(prepared$df$status)
  ))
}

collect_complete_case_summary <- function() {
  rows <- list()
  for (nm in names(endpoint_config)) {
    cfg <- endpoint_config[[nm]]
    for (sp in names(model_specs)) {
      path <- oof_path(cfg$tag, sp)
      if (!file.exists(path)) next
      res <- tryCatch(readRDS(path), error = function(e) {
        warning("Could not read ", path, ": ", conditionMessage(e))
        NULL
      })
      if (is.null(res)) next
      oof <- res$oof_predictions
      rows[[length(rows) + 1L]] <- data.frame(
        endpoint = nm,
        label    = cfg$label,
        tag      = cfg$tag,
        spec     = sp,
        n        = nrow(oof),
        events   = sum(oof$status),
        c_index  = res$c_index,
        se       = res$se,
        ci_lower = res$ci_lower,
        ci_upper = res$ci_upper,
        stringsAsFactors = FALSE
      )
      rm(res)
    }
  }
  dplyr::bind_rows(rows)
}


## How to run

Run the cells above, then the full grid. `skip_existing = TRUE` resumes from the first missing `.rds`. Set it to `FALSE` to refit.

```r
run_endpoint("t2d")
run_endpoint("copd", specs = c("marker", "marker_nmr"))
```


In [ ]:
# Smoke test: T2D, all six specs.
# t2d_complete <- run_endpoint("t2d", skip_existing = TRUE)
# t2d_complete


In [ ]:
# Full grid: 10 endpoints x 6 specs.
failed <- character()
for (nm in names(endpoint_config)) {
  message("\n#### ", nm, " (", endpoint_config[[nm]]$label, ") ####")
  tryCatch(
    run_endpoint(nm, skip_existing = TRUE),
    error = function(e) {
      failed <<- c(failed, nm)
      warning("Failed ", nm, ": ", conditionMessage(e))
      NULL
    }
  )
}
if (length(failed) > 0) {
  warning("Failed endpoints: ", paste(failed, collapse = ", "))
} else {
  message("All endpoints finished (or were already on disk).")
}


## Absolute C-index

COPD is fit and included in the cohort summary, and omitted from the figure and the wide table.


In [ ]:
source(here("helpers", "plot_oof_cindex_primary.r"))

FIG_PANEL_ORDER <- c(
  "t2d", "ckd", "masld",
  "hf_cm", "acute_mi", "chd_revasc",
  "tia_stroke", "total_cvd", "afib"
)

SPEC_MAP <- data.frame(
  spec  = c("base", "base_nmr", "clin", "clin_nmr", "marker", "marker_nmr"),
  model = c("base", "base", "clin", "clin", "marker", "marker"),
  type  = c("SOC", "NMR", "SOC", "NMR", "SOC", "NMR"),
  stringsAsFactors = FALSE
)

endpoint_cindex_ready <- function(df_ep) {
  nrow(df_ep) > 0 &&
    identical(sort(unique(df_ep$spec)), sort(names(model_specs)))
}

write_cindex_csvs <- function(summary_df) {
  if (is.null(summary_df) || nrow(summary_df) == 0L) {
    message("No OOF summaries on disk yet.")
    return(invisible(character()))
  }
  written <- character()
  for (nm in unique(summary_df$endpoint)) {
    df_ep <- summary_df[summary_df$endpoint == nm, , drop = FALSE]
    if (!endpoint_cindex_ready(df_ep)) {
      message("Skipping CSV (incomplete specs): ", nm)
      next
    }
    tag <- unique(df_ep$tag)
    path <- file.path(COMPLETE_CASE_ROOT, "c_index_summaries", paste0(tag, "_cindex_ci.csv"))
    write.csv(prep_cindex_dumbbell_from_specs(df_ep), path, row.names = FALSE)
    written <- c(written, path)
    message("Wrote ", path)
  }
  invisible(written)
}

fully_tested_subtitle <- function(df_ep) {
  # n / events are identical across specs for a given endpoint cohort
  row <- df_ep[df_ep$spec == "base", , drop = FALSE][1, ]
  n_events <- row$events
  n_censored <- row$n - n_events
  sprintf(
    "Fully tested N = %s / %s",
    format(n_events, big.mark = ","),
    format(n_censored, big.mark = ",")
  )
}

plot_cindex_complete_case <- function(summary_df, xlim = c(0.5, 0.85)) {
  plots <- list()
  missing <- character()

  for (nm in FIG_PANEL_ORDER) {
    df_ep <- summary_df[summary_df$endpoint == nm, , drop = FALSE]
    if (!endpoint_cindex_ready(df_ep)) {
      missing <- c(missing, nm)
      next
    }
    plots[[nm]] <- plot_oof_cindex(
      df = prep_cindex_dumbbell_from_specs(df_ep),
      title = endpoint_config[[nm]]$plot_label,
      subtitle = fully_tested_subtitle(df_ep),
      xlim = xlim,
      marker_label = "SOC",
      type_labels = c("SOC" = "No NMR", "NMR" = "NMR")
    )
  }

  if (length(missing) > 0) message("Missing panels: ", paste(missing, collapse = ", "))
  if (length(plots) == 0L) {
    warning("No complete endpoints to plot")
    return(invisible(NULL))
  }

  hide_legend <- function(p) p + theme(legend.position = "none")
  show_bottom <- function(p) p + theme(legend.position = "bottom")

  if (length(plots) == 9L && identical(names(plots), FIG_PANEL_ORDER)) {
    p_all <-
      (hide_legend(plots$t2d) | hide_legend(plots$ckd) | hide_legend(plots$masld)) /
      (hide_legend(plots$hf_cm) | hide_legend(plots$acute_mi) | hide_legend(plots$chd_revasc)) /
      (hide_legend(plots$tia_stroke) | show_bottom(plots$total_cvd) | hide_legend(plots$afib))
  } else {
    plots_adj <- lapply(names(plots), function(nm) {
      if (identical(nm, "total_cvd")) show_bottom(plots[[nm]]) else hide_legend(plots[[nm]])
    })
    p_all <- patchwork::wrap_plots(plots_adj, ncol = 3)
  }

  p_all + patchwork::plot_annotation(
    title = "Complete Case Sensitivity",
    theme = ggplot2::theme(plot.title = ggplot2::element_text(hjust = 0.5))
  )
}

wide_cindex_table <- function(summary_df) {
  endpoint_order <- c(
    "T2D", "CKD", "MASLD",
    "Heart Failure", "Acute MI", "CHD",
    "TIA/Stroke", "Total CVD", "Afib"
  )
  col_order <- c("endpoint", "BASE", "BASE+NMR", "CLIN", "CLIN+NMR", "SOC", "SOC+NMR")

  summary_df %>%
    filter(endpoint %in% FIG_PANEL_ORDER) %>%
    left_join(SPEC_MAP, by = "spec") %>%
    mutate(
      endpoint = factor(label, levels = endpoint_order),
      col = case_when(
        model == "base"   & type == "SOC" ~ "BASE",
        model == "base"   & type == "NMR" ~ "BASE+NMR",
        model == "clin"   & type == "SOC" ~ "CLIN",
        model == "clin"   & type == "NMR" ~ "CLIN+NMR",
        model == "marker" & type == "SOC" ~ "SOC",
        model == "marker" & type == "NMR" ~ "SOC+NMR",
        TRUE ~ NA_character_
      ),
      value = sprintf("%.3f (%.3f, %.3f)", c_index, ci_lower, ci_upper)
    ) %>%
    filter(!is.na(col)) %>%
    select(endpoint, col, value) %>%
    tidyr::pivot_wider(names_from = col, values_from = value) %>%
    arrange(endpoint) %>%
    select(any_of(col_order))
}


In [ ]:
complete_case_summary <- collect_complete_case_summary()
print(complete_case_summary)

write.csv(
  complete_case_summary,
  file.path(COMPLETE_CASE_ROOT, "cohort_cindex_summary.csv"),
  row.names = FALSE
)
write_cindex_csvs(complete_case_summary)

cindex_table <- wide_cindex_table(complete_case_summary)
cindex_table
write.csv(
  cindex_table,
  here("outputs/tables", "CINDEX_STATS_COMPLETE_CASE.csv"),
  row.names = FALSE
)

options(repr.plot.height = 5, repr.plot.width = 7)
p_complete <- plot_cindex_complete_case(complete_case_summary)
p_complete

ggsave(
  file.path(FIG_ROOT, "COMPLETE_CASE.png"),
  plot = p_complete,
  width = 7,
  height = 5,
  dpi = 600
)


# Delta C-index

Paired compareC tests from the saved complete-case OOF files. COPD is omitted from the figure.


In [ ]:
library(ggh4x)
source(here("helpers", "delta_cindex_oof.r"))

DELTA_ENDPOINTS <- endpoint_config[c(
  "t2d", "ckd", "masld",
  "hf_cm", "acute_mi", "chd_revasc",
  "tia_stroke", "total_cvd", "afib"
)]

DELTA_SPEC_MAP <- c(
  base       = "base_SOC",
  base_nmr   = "base_NMR",
  clin       = "clin_SOC",
  clin_nmr   = "clin_NMR",
  marker     = "marker_SOC",
  marker_nmr = "marker_NMR"
)

DELTA_COMPARISONS <- data.frame(
  model_1 = c("base_SOC",   "clin_SOC",   "marker_SOC", "marker_SOC"),
  model_2 = c("base_NMR",   "clin_NMR",   "marker_NMR", "clin_NMR"),
  stringsAsFactors = FALSE
)

complete_case_delta_path <- function(tag) {
  file.path(COMPLETE_CASE_ROOT, "c_index_summaries", paste0(tag, "_delta_cindex.csv"))
}

load_complete_case_oof_results <- function(tag) {
  results <- lapply(names(DELTA_SPEC_MAP), function(sp) {
    path <- oof_path(tag, sp)
    if (!file.exists(path)) stop("Missing complete-case OOF: ", path)
    strip_oof_result(readRDS(path))
  })
  names(results) <- unname(DELTA_SPEC_MAP)
  results
}

run_complete_case_delta <- function(nm,
                                    skip_existing = TRUE,
                                    parallel = FALSE,
                                    workers = 2L) {
  cfg <- DELTA_ENDPOINTS[[nm]]
  if (is.null(cfg)) stop("Unknown delta endpoint: ", nm)

  path <- complete_case_delta_path(cfg$tag)
  if (skip_existing && file.exists(path)) {
    message("Skipping existing delta CSV: ", basename(path))
    return(
      read.csv(path, stringsAsFactors = FALSE) %>%
        mutate(endpoint = cfg$label, delta_endpoint = nm, tag = cfg$tag)
    )
  }

  message("Delta C: ", nm, " (", cfg$label, ")")
  results <- load_complete_case_oof_results(cfg$tag)
  delta <- run_delta_cindex_batch(
    results = results,
    comparisons = DELTA_COMPARISONS,
    strip_first = TRUE,
    parallel = parallel,
    workers = workers,
    parallel_transport = "rds",
    close_workers_on_exit = TRUE
  )
  delta$endpoint <- cfg$label
  delta$delta_endpoint <- nm
  delta$tag <- cfg$tag
  write.csv(delta, path, row.names = FALSE)
  message("Wrote ", path)

  rm(results)
  invisible(gc(verbose = FALSE))
  delta
}


In [ ]:
delta_failed <- character()
for (nm in names(DELTA_ENDPOINTS)) {
  tryCatch(
    run_complete_case_delta(nm, skip_existing = TRUE, parallel = FALSE),
    error = function(e) {
      delta_failed <<- c(delta_failed, nm)
      warning("Failed delta C for ", nm, ": ", conditionMessage(e))
      NULL
    }
  )
}
if (length(delta_failed) > 0) {
  warning("Delta C failed for: ", paste(delta_failed, collapse = ", "))
} else {
  message("All complete-case delta C endpoints finished (or were already on disk).")
}


In [ ]:
delta_list <- lapply(names(DELTA_ENDPOINTS), function(nm) {
  cfg <- DELTA_ENDPOINTS[[nm]]
  path <- complete_case_delta_path(cfg$tag)
  if (!file.exists(path)) {
    message("Missing delta CSV: ", basename(path))
    return(NULL)
  }
  read.csv(path, stringsAsFactors = FALSE) %>%
    mutate(endpoint = cfg$label, delta_endpoint = nm, tag = cfg$tag)
})
names(delta_list) <- names(DELTA_ENDPOINTS)

delta_all <- dplyr::bind_rows(delta_list) %>%
  mutate(
    boost = case_when(
      model_1 == "base_SOC"   & model_2 == "base_NMR"   ~ "BASE",
      model_1 == "clin_SOC"   & model_2 == "clin_NMR"   ~ "CLIN",
      model_1 == "marker_SOC" & model_2 == "marker_NMR" ~ "SOC",
      model_1 == "marker_SOC" & model_2 == "clin_NMR"   ~ "CLIN+NMR vs SOC",
      TRUE ~ NA_character_
    )
  ) %>%
  filter(!is.na(boost)) %>%
  group_by(boost) %>%
  mutate(p_adj_BH = p.adjust(p_value, method = "BH")) %>%
  ungroup()

delta_all

write.csv(
  delta_all,
  file.path(COMPLETE_CASE_ROOT, "c_index_summaries", "complete_case_delta_cindex_summary.csv"),
  row.names = FALSE
)


In [ ]:
plot_delta_c_boost <- function(
    df,
    x_max = 0.25,
    x_by = 0.10,
    x_min_comparison = -0.02,
    annotation_pad = 0.025
) {
  endpoint_order <- c(
    "T2D", "CKD", "MASLD",
    "Heart Failure", "Acute MI", "CHD",
    "TIA/Stroke", "Total CVD", "Afib"
  )
  boost_order <- c("BASE", "CLIN", "SOC", "CLIN+NMR vs SOC")

  plot_df <- df %>%
    mutate(
      boost = case_when(
        model_1 == "base_SOC"   & model_2 == "base_NMR"   ~ "BASE",
        model_1 == "clin_SOC"   & model_2 == "clin_NMR"   ~ "CLIN",
        model_1 == "marker_SOC" & model_2 == "marker_NMR" ~ "SOC",
        model_1 == "marker_SOC" & model_2 == "clin_NMR"   ~ "CLIN+NMR vs SOC",
        TRUE ~ NA_character_
      ),
      boost = factor(boost, levels = boost_order),
      endpoint = factor(endpoint, levels = rev(endpoint_order))
    ) %>%
    filter(!is.na(boost)) %>%
    mutate(
      sig_x = if_else(delta_c >= 0, ci_upper + 0.004, ci_lower - 0.004),
      sig_hjust = if_else(delta_c >= 0, 0, 1),
      sig_label = case_when(
        is.na(p_adj_BH)  ~ NA_character_,
        p_adj_BH >= 0.05 ~ "n.s.",
        p_adj_BH < 0.001 ~ "***",
        p_adj_BH < 0.01  ~ "**",
        TRUE             ~ "*"
      )
    )

  short_x_labels <- function(x) {
    vapply(
      x,
      function(z) {
        if (abs(z) < 1e-10) return("0")
        if (z < 0) return(paste0("-", sub("^0", "", sprintf("%.2f", abs(z)))))
        sub("^0", "", sprintf("%.2f", z))
      },
      character(1)
    )
  }

  main_breaks <- seq(0, floor(x_max / x_by) * x_by, by = x_by)

  ggplot(plot_df, aes(x = delta_c, y = endpoint)) +
    geom_col(width = 0.62, fill = "steelblue") +
    geom_errorbarh(aes(xmin = ci_lower, xmax = ci_upper), height = 0.18, linewidth = 0.45) +
    geom_text(
      data = plot_df %>% filter(!is.na(sig_label)),
      aes(x = sig_x, label = sig_label, hjust = sig_hjust),
      size = 2.7,
      vjust = 0.7
    ) +
    geom_vline(xintercept = 0, linewidth = 0.25, colour = "black") +
    facet_wrap(~ boost, nrow = 1, scales = "free_x") +
    ggh4x::facetted_pos_scales(
      x = list(
        boost %in% c("BASE", "CLIN") ~
          scale_x_continuous(
            limits = c(0, x_max + annotation_pad),
            breaks = main_breaks,
            labels = short_x_labels,
            expand = expansion(mult = c(0, 0))
          ),
        boost %in% c("SOC", "CLIN+NMR vs SOC") ~
          scale_x_continuous(
            limits = c(x_min_comparison, x_max + annotation_pad),
            breaks = seq(0, x_max, by = x_by),
            labels = short_x_labels,
            expand = expansion(mult = c(0, 0))
          )
      )
    ) +
    scale_y_discrete(drop = FALSE, expand = expansion(add = c(0.50, 0.55))) +
    labs(
      x = expression(Delta~"C-index (95% CI)"),
      y = NULL,
      title = "Incremental NMR Improvement (Complete cases)"
    ) +
    theme_bw(base_size = 11) +
    theme(
      strip.background = element_rect(fill = "grey92", colour = NA),
      strip.text = element_text(face = "plain", size = 9.5, margin = margin(t = 4, r = 2, b = 4, l = 2)),
      panel.border = element_blank(),
      panel.grid.major.y = element_blank(),
      panel.grid.major.x = element_line(colour = "grey90", linewidth = 0.3),
      panel.grid.minor = element_blank(),
      panel.spacing.x = grid::unit(0.35, "cm"),
      axis.line.x = element_line(colour = "black", linewidth = 0.4),
      axis.ticks.x = element_line(colour = "black", linewidth = 0.35),
      axis.ticks.y = element_blank(),
      axis.text.x = element_text(size = 8, colour = "black", margin = margin(t = 2)),
      axis.text.y = element_text(size = 8.7, colour = "black", margin = margin(r = 3)),
      axis.title.x = element_text(size = 10.5, margin = margin(t = 5)),
      plot.title = element_text(hjust = 0.5, face = "plain", size = 12, margin = margin(b = 6)),
      plot.margin = margin(t = 3, r = 5, b = 2, l = 3)
    )
}


In [ ]:
options(repr.plot.height = 2.15, repr.plot.width = 7)

p_delta_complete <- plot_delta_c_boost(
  df = delta_all,
  x_max = 0.25,
  x_by = 0.10,
  x_min_comparison = -0.07,
  annotation_pad = 0.025
)
p_delta_complete

ggsave(
  file.path(FIG_ROOT, "COMPLETE_CASE_delta.png"),
  plot = p_delta_complete,
  width = 7,
  height = 2.5,
  dpi = 300
)
